# Day 061 — Exercise 3: MetricsCollector

Logging tells you WHAT happened. Metrics tell you HOW OFTEN and HOW FAST. The key counters for an AI API are:

- **request count** — total traffic
- **error count** — 4xx/5xx responses
- **error rate** — fraction of requests that failed
- **average latency** — how long requests take on average

A single in-memory accumulator is sufficient for one server instance. For multi-instance deployments, metrics would be pushed to a central store (Prometheus, Datadog) — the same interface, different backend.

In [ ]:
# no extra imports needed


## Task

Implement `MetricsCollector`:

| Method | Behaviour |
|--------|-----------|
| `record(status_code, duration_ms)` | Increment requests; if `status_code >= 400` also increment errors; append duration_ms |
| `summary() -> dict` | `{requests, errors, avg_latency_ms (1dp), error_rate (3dp)}` |
| `reset()` | Clear all counters and latency list |

`error_rate = errors / requests` (or `0.0` when no requests).

## Your Implementation

In [ ]:
class MetricsCollector:
    """Accumulate per-request metrics.

    record(status_code, duration_ms) — add one request observation.
        status_code >= 400 counts as an error.
    summary() -> dict — return aggregate stats:
        {"requests": int, "errors": int,
         "avg_latency_ms": float (1 dp), "error_rate": float (3 dp)}
        error_rate = errors / requests (0.0 if no requests)
        avg_latency_ms = 0.0 if no requests
    reset() — clear all counters and samples.
    """

    def __init__(self):
        # TODO: init counters and latency list
        raise NotImplementedError

    def record(self, status_code: int, duration_ms: float) -> None:
        raise NotImplementedError

    def summary(self) -> dict:
        raise NotImplementedError

    def reset(self) -> None:
        raise NotImplementedError


In [ ]:
class MetricsCollector:
    def __init__(self):
        self._requests = 0
        self._errors   = 0
        self._latencies: list = []

    def record(self, status_code: int, duration_ms: float) -> None:
        self._requests += 1
        if status_code >= 400:
            self._errors += 1
        self._latencies.append(duration_ms)

    def summary(self) -> dict:
        avg        = sum(self._latencies) / len(self._latencies) if self._latencies else 0.0
        error_rate = self._errors / self._requests if self._requests else 0.0
        return {
            "requests":       self._requests,
            "errors":         self._errors,
            "avg_latency_ms": round(avg, 1),
            "error_rate":     round(error_rate, 3),
        }

    def reset(self) -> None:
        self._requests = 0
        self._errors   = 0
        self._latencies.clear()


## Automated checks

In [ ]:
score, total = 0, 6
try:
    mc = MetricsCollector()

    # empty summary
    s0 = mc.summary()
    assert s0["requests"] == 0 and s0["errors"] == 0
    assert s0["avg_latency_ms"] == 0.0 and s0["error_rate"] == 0.0
    score += 1; print("\u2705 empty collector returns zero summary")

    # record a 200
    mc.record(200, 50.0)
    s1 = mc.summary()
    assert s1["requests"] == 1 and s1["errors"] == 0
    assert s1["error_rate"] == 0.0
    score += 1; print("\u2705 200 response: requests=1, errors=0")

    # record a 404 (error)
    mc.record(404, 10.0)
    s2 = mc.summary()
    assert s2["requests"] == 2 and s2["errors"] == 1
    score += 1; print("\u2705 404 counts as an error")

    # record a 500 (also error)
    mc.record(500, 30.0)
    s3 = mc.summary()
    assert s3["errors"] == 2 and s3["requests"] == 3
    score += 1; print("\u2705 500 counts as an error")

    # avg_latency_ms (50 + 10 + 30) / 3 = 30.0
    assert s3["avg_latency_ms"] == 30.0, (
        f"Expected 30.0, got {s3['avg_latency_ms']}")
    score += 1; print("\u2705 avg_latency_ms is correct")

    # error_rate = 2/3 ≈ 0.667
    assert abs(s3["error_rate"] - 0.667) < 0.001, (
        f"Expected ~0.667, got {s3['error_rate']}")
    # reset
    mc.reset()
    assert mc.summary()["requests"] == 0
    score += 1; print("\u2705 reset() clears all counters")

except Exception as e:
    print(f"\u274c {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
class MetricsCollector:
    def __init__(self):
        self._requests = 0
        self._errors   = 0
        self._latencies: list = []

    def record(self, status_code: int, duration_ms: float) -> None:
        self._requests += 1
        if status_code >= 400:
            self._errors += 1
        self._latencies.append(duration_ms)

    def summary(self) -> dict:
        avg        = sum(self._latencies) / len(self._latencies) if self._latencies else 0.0
        error_rate = self._errors / self._requests if self._requests else 0.0
        return {
            "requests":       self._requests,
            "errors":         self._errors,
            "avg_latency_ms": round(avg, 1),
            "error_rate":     round(error_rate, 3),
        }

    def reset(self) -> None:
        self._requests = 0
        self._errors   = 0
        self._latencies.clear()
```

**Latency list instead of a running sum?** A list lets you compute percentiles (p50, p95, p99) later. A running sum only gives you the mean. In production you'd use a histogram or reservoir sampling to keep memory bounded — but for a single-server AI app, a plain list is fine.

</details>